# Extracción de huellas de edificios

## 1. Configuración del entorno

Se importan las librerías necesarias para la fase final. Además de `geoai`, se incorporan `gdal` para el manejo avanzado de rásters y `gc` (Garbage Collector) para la gestión eficiente de la memoria RAM durante el procesamiento de imágenes de gran tamaño.

In [ ]:
import geoai
import os
from osgeo import gdal
import gc
import shutil
# Configurar rutas relativas
PROCESSED_DATA_PATH = "../data/processed/raster"
MODELS_DATA_PATH = "../data/geoai/models"
OUTPUT_PATH = "../data/geoai/results"

## 2. Segmentación de los casos de estudio

Se ejecuta el proceso de inferencia sobre la ortofoto completa de 2021 (pre-desastre). Para evitar desbordamientos de memoria, la imagen original se divide en cuatro cuadrantes mediante `gdal.Translate`. Se aplica el modelo seleccionado (**U-Net + EfficientNet-B1**) sobre cada cuadrante de forma secuencial. Finalmente, las predicciones parciales se ensamblan nuevamente en un único mosaico utilizando `gdal.BuildVRT` y `gdal.Translate`, generando el mapa de probabilidad de edificios para toda la zona de estudio.

In [ ]:
# Configurar rutas temporales
INPUT_RASTER = f"{PROCESSED_DATA_PATH}/orto_2021_clipped.tif"
TEMP_DIR = f"{PROCESSED_DATA_PATH}/temp_tiles_2021"
FINAL_OUTPUT = f"{OUTPUT_PATH}/orto_2021_prediction.tif"
os.makedirs(TEMP_DIR, exist_ok=True) # Crear directorio temporal si se ha borrado al final del proceso

# Preparar cuadrantes para reducir coste computacional
print(f'Analizando imagen base: {os.path.basename(INPUT_RASTER)}...')
ds = gdal.Open(INPUT_RASTER)
width = ds.RasterXSize
height = ds.RasterYSize
mid_x, mid_y = width // 2, height // 2

tiles_config = {
    'tile_tl': {'off_x': 0,     'off_y': 0,     'sx': mid_x,       'sy': mid_y},        # Arriba a la izquierda
    'tile_tr': {'off_x': mid_x, 'off_y': 0,     'sx': width-mid_x, 'sy': mid_y},        # Arriba a la derecha
    'tile_bl': {'off_x': 0,     'off_y': mid_y, 'sx': mid_x,       'sy': height-mid_y}, # Abajo a la izquierda
    'tile_br': {'off_x': mid_x, 'off_y': mid_y, 'sx': width-mid_x, 'sy': height-mid_y}  # Abajo a la derecha
}

prediction_files = []

# Recortar y aplicar la inferencia a cada cuadrante
for name, coords in tiles_config.items():
    print(f'  -> Procesando cuadrante: {name}...')
    
    tile_src = f"{TEMP_DIR}/{name}_src.tif"
    tile_pred = f"{TEMP_DIR}/{name}_pred.tif"
    
    gdal.Translate(
        tile_src, 
        ds, 
        srcWin=[coords['off_x'], coords['off_y'], coords['sx'], coords['sy']]
    )
    
    # Inferencia con GeoAI (2021)
    geoai.semantic_segmentation(
        input_path=tile_src,
        output_path=tile_pred,
        model_path=f"{MODELS_DATA_PATH}/unet_efficientnet/best_model.pth",
        architecture="unet",
        encoder_name="efficientnet-b1",
        num_channels=3,
        num_classes=2,
        window_size=512,
        overlap=256,   
        batch_size=8,
        probability_threshold=0.3,
        quiet=False,
    )
    
    prediction_files.append(tile_pred)
    
    # Limpiar memoria antes de la siguiente ventana
    gc.collect()

ds = None

# Unión de las ventanas
print(f'Generando mosaico final: {os.path.basename(FINAL_OUTPUT)}...')
vrt_path = f"{TEMP_DIR}/merged_temp.vrt"
gdal.BuildVRT(vrt_path, prediction_files)

creation_options = [
    "TILED=YES",
    "BLOCKXSIZE=512",
    "BLOCKYSIZE=512",
    "COMPRESS=LZW",
    "PREDICTOR=2",
    "BIGTIFF=IF_NEEDED",
    "NUM_THREADS=ALL_CPUS"
]

gdal.Translate(
    FINAL_OUTPUT,
    vrt_path,
    creationOptions=creation_options,
    callback=gdal.TermProgress_nocb 
)

print(f'\n  -> Proceso finalizado con éxito. Segmentación disponible en {FINAL_OUTPUT}')

# Limpiar temporales (opcional)
shutil.rmtree(TEMP_DIR)

Analizando imagen base: orto_2021_clipped.tif...
  -> Procesando cuadrante: tile_tl...
Input file format: GeoTIFF (.tif)
Processing 3744 windows...


3871it [01:18, 49.62it/s]                                                                           


Using probability threshold: 0.3
Predicted classes: 2 classes, Background: 97.8%
Inference completed in 93.83 seconds
Saved prediction to ../data/processed/raster/temp_tiles_2021/tile_tl_pred.tif
  -> Procesando cuadrante: tile_tr...
Input file format: GeoTIFF (.tif)
Processing 3744 windows...


3871it [01:16, 50.52it/s]                                                                           


Using probability threshold: 0.3
Predicted classes: 2 classes, Background: 98.2%
Inference completed in 92.47 seconds
Saved prediction to ../data/processed/raster/temp_tiles_2021/tile_tr_pred.tif
  -> Procesando cuadrante: tile_bl...
Input file format: GeoTIFF (.tif)
Processing 3744 windows...


3871it [01:17, 49.88it/s]                                                                           


Using probability threshold: 0.3
Predicted classes: 2 classes, Background: 99.1%
Inference completed in 93.41 seconds
Saved prediction to ../data/processed/raster/temp_tiles_2021/tile_bl_pred.tif
  -> Procesando cuadrante: tile_br...
Input file format: GeoTIFF (.tif)
Processing 3744 windows...


3871it [01:16, 50.30it/s]                                                                           


Using probability threshold: 0.3
Predicted classes: 2 classes, Background: 98.6%
Inference completed in 92.85 seconds
Saved prediction to ../data/processed/raster/temp_tiles_2021/tile_br_pred.tif
Generando mosaico final: orto_2021_prediction.tif...
100 - done.
0...10...20...30...40...50...60...70...80...90...
  -> Proceso finalizado con éxito. Segmentación disponible en ../data/geoai/results/orto_2021_prediction.tif


Se repite el procedimiento de segmentación semántica para la ortofoto de 2024 (post-desastre). La comparación entre este resultado y el de 2021 permitirá identificar los edificios desaparecidos. El uso del mismo modelo y parámetros garantiza la consistencia en la detección de cambios.

In [ ]:
# Configurar rutas temporales
INPUT_RASTER = f"{PROCESSED_DATA_PATH}/orto_2024_clipped.tif"
TEMP_DIR = f"{PROCESSED_DATA_PATH}/temp_tiles_2024"
FINAL_OUTPUT = f"{OUTPUT_PATH}/orto_2024_prediction.tif"
os.makedirs(TEMP_DIR, exist_ok=True) # Crear directorio temporal si se ha borrado al final del proceso

# Preparar cuadrantes para reducir coste computacional
print(f'Analizando imagen base: {os.path.basename(INPUT_RASTER)}...')
ds = gdal.Open(INPUT_RASTER)
width = ds.RasterXSize
height = ds.RasterYSize
mid_x, mid_y = width // 2, height // 2

tiles_config = {
    'tile_tl': {'off_x': 0,     'off_y': 0,     'sx': mid_x,       'sy': mid_y},        # Arriba a la izquierda
    'tile_tr': {'off_x': mid_x, 'off_y': 0,     'sx': width-mid_x, 'sy': mid_y},        # Arriba a la derecha
    'tile_bl': {'off_x': 0,     'off_y': mid_y, 'sx': mid_x,       'sy': height-mid_y}, # Abajo a la izquierda
    'tile_br': {'off_x': mid_x, 'off_y': mid_y, 'sx': width-mid_x, 'sy': height-mid_y}  # Abajo a la derecha
}

prediction_files = []

# Recortar y aplicar la inferencia a cada cuadrante
for name, coords in tiles_config.items():
    print(f'  -> Procesando cuadrante: {name}...')
    
    tile_src = f"{TEMP_DIR}/{name}_src.tif"
    tile_pred = f"{TEMP_DIR}/{name}_pred.tif"
    
    gdal.Translate(
        tile_src, 
        ds, 
        srcWin=[coords['off_x'], coords['off_y'], coords['sx'], coords['sy']]
    )
    
    # Inferencia con GeoAI (2024)
    geoai.semantic_segmentation(
        input_path=tile_src,
        output_path=tile_pred,
        model_path=f"{MODELS_DATA_PATH}/unet_efficientnet/best_model.pth",
        architecture="unet",
        encoder_name="efficientnet-b1",
        num_channels=3,
        num_classes=2,
        window_size=512,
        overlap=256,   
        batch_size=8,
        probability_threshold=0.3,
        quiet=False,
    )
    
    prediction_files.append(tile_pred)
    
    # Limpiar memoria antes de la siguiente ventana
    gc.collect()

ds = None

# Unión de las ventanas
print(f'Generando mosaico final: {os.path.basename(FINAL_OUTPUT)}...')
vrt_path = f"{TEMP_DIR}/merged_temp.vrt"
gdal.BuildVRT(vrt_path, prediction_files)

creation_options = [
    "TILED=YES",
    "BLOCKXSIZE=512",
    "BLOCKYSIZE=512",
    "COMPRESS=LZW",
    "PREDICTOR=2",
    "BIGTIFF=IF_NEEDED",
    "NUM_THREADS=ALL_CPUS"
]

gdal.Translate(
    FINAL_OUTPUT,
    vrt_path,
    creationOptions=creation_options,
    callback=gdal.TermProgress_nocb 
)

print(f'\n  -> Proceso finalizado con éxito. Segmentación disponible en {FINAL_OUTPUT}')

# Limpiar temporales (opcional)
shutil.rmtree(TEMP_DIR)

Analizando imagen base: orto_2024_clipped.tif...
  -> Procesando cuadrante: tile_tl...
Input file format: GeoTIFF (.tif)
Processing 3744 windows...


3871it [01:17, 49.65it/s]                                                                           


Using probability threshold: 0.3
Predicted classes: 2 classes, Background: 98.8%
Inference completed in 93.79 seconds
Saved prediction to ../data/processed/raster/temp_tiles_2024/tile_tl_pred.tif
  -> Procesando cuadrante: tile_tr...
Input file format: GeoTIFF (.tif)
Processing 3744 windows...


3871it [01:16, 50.75it/s]                                                                           


Using probability threshold: 0.3
Predicted classes: 2 classes, Background: 99.0%
Inference completed in 92.09 seconds
Saved prediction to ../data/processed/raster/temp_tiles_2024/tile_tr_pred.tif
  -> Procesando cuadrante: tile_bl...
Input file format: GeoTIFF (.tif)
Processing 3744 windows...


3871it [01:18, 49.53it/s]                                                                           


Using probability threshold: 0.3
Predicted classes: 2 classes, Background: 99.6%
Inference completed in 93.96 seconds
Saved prediction to ../data/processed/raster/temp_tiles_2024/tile_bl_pred.tif
  -> Procesando cuadrante: tile_br...
Input file format: GeoTIFF (.tif)
Processing 3744 windows...


3871it [01:16, 50.56it/s]                                                                           


Using probability threshold: 0.3
Predicted classes: 2 classes, Background: 99.1%
Inference completed in 92.40 seconds
Saved prediction to ../data/processed/raster/temp_tiles_2024/tile_br_pred.tif
Generando mosaico final: orto_2024_prediction.tif...
100 - done.
0...10...20...30...40...50...60...70...80...90...
  -> Proceso finalizado con éxito. Segmentación disponible en ../data/geoai/results/orto_2024_prediction.tif


## 3.  Vectorización de las máscaras

Las máscaras de predicción ráster resultantes (donde cada píxel indica la probabilidad de ser edificio) se convierten a formato vectorial. Se utiliza la función `orthogonalize` para transformar los grupos de píxeles en polígonos geoespaciales. Adicionalmente, se calculan y añaden propiedades geométricas a cada polígono, importantes para el filtrado posterior de ruido.

In [ ]:
# Vectorización inicial de la predicción de 2021
RASTER_PATH_2021 = f"{OUTPUT_PATH}/orto_2021_prediction.tif"
VECTOR_PATH_2021 = f"{OUTPUT_PATH}/orto_2021_prediction.geojson"

gdf = geoai.orthogonalize(RASTER_PATH_2021, VECTOR_PATH_2021, epsilon=2)
vector_2021_props = geoai.add_geometric_properties(gdf, area_unit="m2", length_unit="m")

Processing 12616 features...


Converting features: 100%|██████████████████████████████████████████████████████████████████████████████████| 12616/12616 [00:46<00:00, 274.03shape/s]


Saving to ../data/geoai/results/orto_2021_prediction.geojson...
Done!


Se aplica el mismo proceso de vectorización y cálculo de propiedades geométricas a la predicción de 2024.

In [ ]:
# Vectorización inicial de la predicción de 2021
RASTER_PATH_2024 = f"{OUTPUT_PATH}/orto_2024_prediction.tif"
VECTOR_PATH_2024 = f"{OUTPUT_PATH}/orto_2024_prediction.geojson"

gdf = geoai.orthogonalize(RASTER_PATH_2024, VECTOR_PATH_2024, epsilon=2)
vector_2024_props = geoai.add_geometric_properties(gdf, area_unit="m2", length_unit="m")

Processing 7149 features...


Converting features: 100%|████████████████████████████████████████████████████████████████████████████████████| 7149/7149 [00:26<00:00, 267.68shape/s]


Saving to ../data/geoai/results/orto_2024_prediction.geojson...
Done!


## 4. Filtrado y regularización

Se ejecuta una etapa de limpieza y refinamiento de datos. Primero, se aplica un filtro espacial para eliminar polígonos con un área inferior a $15\ m^2$, descartando así falsos positivos causados por ruido en la imagen (vehículos, piscinas, etc.). Posteriormente, se aplica el algoritmo `adaptive_regularization` de GeoAI, que corrige el efecto de dientes de sierra, simplificando la geometría de los edificios para obtener contornos más rectos y simples.

In [ ]:
# Filtro de figuras de menos de 15 metros cuadrados
vector_2021_filter = vector_2021_props[(vector_2021_props["area_m2"] > 15)]
vector_2024_filter = vector_2024_props[(vector_2024_props["area_m2"] > 15)]

# Regularización de las formas de las huellas
area_estudio_2021 = geoai.adaptive_regularization(
    vector_2021_filter, simplify_tolerance=0.5, area_threshold=0.85, preserve_shape=True
)
area_estudio_2024 = geoai.adaptive_regularization(
    vector_2024_filter, simplify_tolerance=0.5, area_threshold=0.85, preserve_shape=True
)

Los vectores finales, ya filtrados y regularizados, se exportan a formato GeoJSON. Estos archivos constituyen el producto final del análisis y están listos para ser integrados en sistemas SIG o visualizadores web.

In [ ]:
# Guardado de las huellas
geoai.add_geometric_properties(
    area_estudio_2021, area_unit="m2", length_unit="m"
).to_file(f"{OUTPUT_PATH}/area_estudio_2021.geojson", driver="GeoJSON")
geoai.add_geometric_properties(
    area_estudio_2024, area_unit="m2", length_unit="m"
).to_file(f"{OUTPUT_PATH}/area_estudio_2024.geojson", driver="GeoJSON")

## 5. Comprobación de la huella final

Como punto final, se genera un mapa interactivo comparativo utilizando `leafmap`. Se cargan las capas vectoriales resultantes de 2021 y 2024 sobre sus respectivas ortofotos base, servidas vía WMS. Esto permite una inspección visual detallada de los resultados, facilitando la validación cualitativa desde el propio cuaderno.

In [ ]:
# Visualización final de las huellas extraidas previo y posterior al desastre
import ipywidgets as w
import leafmap.leafmap as leafmap

data = [
    (
        area_estudio_2021,
        "orange",
        "Predicción reglarizada 2021",
        "https://idecan1.grafcan.es/ServicioWMS/Historico/Ortofotos/OrtoExpress_2021?",
        "IDECanarias OrtoExpress 2021 (20 cm/pixel)",
    ),
    (
        area_estudio_2024,
        "red",
        "Predicción regularizada 2024",
        "https://idecan1.grafcan.es/ServicioWMS/OrtoExpress?",
        "IDECanarias Ortofoto Territorial Campaña 2024",
    ),
]

maps = [
    leafmap.Map(
        center=[28.6162, -17.8986],
        zoom=18,
        min_zoom=16,
        max_zoom=22,
        toolbar_control=False,
        draw_control=False,
        fullscreen_control=False,
        layers_control=True,
    )
    for i in range(2)
]

for m, (geojson, color, model_name, wms_url, wms_layer_name) in zip(maps, data):
    m.clear_layers()

    m.add_wms_layer(
        url=wms_url,
        layers="WMS_OrtoExpress",
        name=wms_layer_name,
        attribution='<a href="https://www.grafcan.es/aviso-legal/" target="_blank">GRAFCAN</a>, Ortofotos Express de Canarias',
        format="image/jpeg",
        max_zoom=22,
        base=True,
    )

    m.add_gdf(
        geojson,
        layer_name=model_name,
        style={"color": color, "fillOpacity": 0.25, "weight": 2},
        zoom_to_layer=False,
    )

w.jslink((maps[0], "center"), (maps[1], "center"))
w.jslink((maps[0], "zoom"), (maps[1], "zoom"))

maps[0].layout.width = maps[1].layout.width = "50%"
display(w.HBox(maps, layout=w.Layout(height="600px")))

Se realiza también un pequeño cálculo cuantitativo directo del impacto del desastre. Se comparan los conteos totales de edificios detectados antes y después de la erupción, proporcionando una estimación numérica preliminar de las estructuras perdidas.

In [ ]:
# Calcular conteo de huellas
num_2021 = len(area_estudio_2021)
num_2024 = len(area_estudio_2024)
diferencia = num_2021 - num_2024

# Imprimir el resultado
print(f"Edificios en 2021: {num_2021}")
print(f"Edificios en 2024: {num_2024}")
print(f"  -> Variación:  {diferencia} edificios perdidos en el desastre")

Edificios en 2021: 5273
Edificios en 2024: 3056
  -> Variación:  2217 edificios perdidos en el desastre
